# fuzzy_matching_demo

Walk through the five-stage pipeline from `fuzzy_matching.py` on four realistic
datasets — **customer names**, **company names**, **addresses**, and **country
names**. The reference lists are the same ones used by the interactive
[fuzzy-matching tool](../web/index.html).

**Author:** Jennifer Payne   
**GitHub:** [@majorpayne-2021](https://github.com/majorpayne-2021)  
**LinkedIn:** [jenniferapayne25](https://www.linkedin.com/in/jenniferapayne25/)  
*Making the complex simple, one tech project at a time.*

---

**What you'll see:**

1. Load messy inputs and an authoritative reference list from CSV.
2. Run each input through `best_match()` — it tries every scorer and keeps the highest score, surfacing which method won.
3. For each scenario, see a full per-method comparison table on a representative input — rows are candidates, columns are scorers, the gradient shows score strength.

Each scenario is independent — re-run any cell to follow the logic.


## Setup

Install missing libraries (silent if already present), then import everything.

In [1]:
%pip install -q rapidfuzz matplotlib

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import sys

import pandas as pd

# Make `fuzzy_matching` importable when this notebook is opened from `examples/`.
sys.path.insert(0, str(Path.cwd()))

from fuzzy_matching import (
    SCORERS,
    best_match,
    classify,
    jaccard_ngrams_score,
    jaccard_tokens_score,
    jaro_winkler_score,
    levenshtein_score,
    normalise,
    standardise_name,
)

FIXTURES = Path("..") / "fixtures"
inputs_df = pd.read_csv(FIXTURES / "inputs.csv")
reference_df = pd.read_csv(FIXTURES / "reference.csv")

print(f"{len(inputs_df):>3} messy inputs across {inputs_df.scenario.nunique()} scenarios")
print(f"{len(reference_df):>3} reference entries")
inputs_df.head()

 29 messy inputs across 4 scenarios
 76 reference entries


,scenario,value,expected_match
0,customer_names,Jhon Smith,John Smith
1,customer_names,Katherine taylor,Katherine Taylor
2,customer_names,jose hernandes,Jose Hernandez
3,customer_names,Mueller Schmit,Mueller Schmidt
4,customer_names,Siobhan OConnor,Siobhan O’Connor


In [3]:
# How many reference entries per scenario?
reference_df.groupby("scenario").size().rename("reference_entries").to_frame()

,reference_entries
scenario,
addresses,14
company_names,16
country_names,24
customer_names,22


## Walkthrough — best-of-all-methods per input

`best_match()` tries every scorer in `SCORERS` against every reference entry and returns the highest-scoring (candidate, method) combination. The `best_method` column tells you which scorer won — that signal alone is informative: it shifts depending on what kind of noise the input has.

In [4]:
def run_scenario(scenario: str) -> pd.DataFrame:
    """Run best_match() on every messy input in `scenario`. Returns a
    DataFrame with the chosen match, score, winning method, and a
    `correct` column comparing against the expected answer."""
    rows = inputs_df.query("scenario == @scenario")
    reference = reference_df.query("scenario == @scenario")["value"].tolist()

    results = []
    for _, row in rows.iterrows():
        out = best_match(row["value"], reference)
        results.append(
            {
                "input": out["input"],
                "matched_to": out["match"],
                "score": round(out["score"], 3),
                "best_method": out["method"],
                "decision": out["decision"],
                "via": out["via"],
                "expected": row["expected_match"],
                "correct": out["match"] == row["expected_match"],
            }
        )
    return pd.DataFrame(results)

### 1. Customer names

Typos, missing letters, casing, word order. Watch which scorer wins most often — it's usually `jaro_winkler` for short name strings.

In [5]:
run_scenario("customer_names")

,input,matched_to,score,best_method,decision,via,expected,correct
0,Jhon Smith,John Smith,0.970,jaro_winkler,auto,fuzzy,John Smith,True
1,Katherine taylor,Katherine Taylor,1.000,exact,auto,exact,Katherine Taylor,True
2,jose hernandes,Jose Hernandez,0.971,jaro_winkler,auto,fuzzy,Jose Hernandez,True
3,Mueller Schmit,Mueller Schmidt,0.987,jaro_winkler,auto,fuzzy,Mueller Schmidt,True
4,Siobhan OConnor,Siobhan O’Connor,0.987,jaro_winkler,auto,fuzzy,Siobhan O’Connor,True
5,francois Dubois,Francois Dubois,1.000,exact,auto,exact,Francois Dubois,True
6,Pryia Patel,Priya Patel,0.976,jaro_winkler,auto,fuzzy,Priya Patel,True
7,wei chen,Wei Chen,1.000,exact,auto,exact,Wei Chen,True


### 2. Company names

Legal-suffix noise (`Pty Ltd`, `Limited`, `Plc`) plus typos. Different inputs win on different methods — `BHP` doesn't behave like `Commwealth Bank Aus`. Note: the default thresholds (`auto=0.90`, `review=0.70`) are tuned for Jaro-Winkler on short strings, so a correct match via Jaccard tokens will often land in the `review` bucket. **Calibrate thresholds per scorer for production.**

In [6]:
run_scenario("company_names")

,input,matched_to,score,best_method,decision,via,expected,correct
0,Commwealth Bank Aus,Commonwealth Bank of Australia,0.874,jaro_winkler,review,fuzzy,Commonwealth Bank of Australia,True
1,BHP,BHP Group Limited,0.808,jaro_winkler,review,fuzzy,BHP Group Limited,True
2,Telstra,Telstra Group Limited,0.867,jaro_winkler,review,fuzzy,Telstra Group Limited,True
3,Qantas Pty Ltd,Qantas Airways Limited,0.881,jaro_winkler,review,fuzzy,Qantas Airways Limited,True
4,Macquarie group,Macquarie Group Limited,0.930,jaro_winkler,auto,fuzzy,Macquarie Group Limited,True
5,Wollworths Group,Woolworths Group Limited,0.794,jaro_winkler,review,fuzzy,Woolworths Group Limited,True
6,atlassian corp,Atlassian Corporation Plc,0.912,jaro_winkler,auto,fuzzy,Atlassian Corporation Plc,True


### 3. Addresses

Long strings with abbreviations (`St` ↔ `Street`), embedded numbers, and a postcode tail. n-gram overlap usually wins here, since it catches partial-word matches that token-set and edit-distance can't.

In [7]:
run_scenario("addresses")

,input,matched_to,score,best_method,decision,via,expected,correct
0,130 Swanson st melborne VIC,"130 Swanston Street, Melbourne VIC 3000",0.898,jaro_winkler,review,fuzzy,"130 Swanston Street, Melbourne VIC 3000",True
1,Parliament Hse Canberra,"Parliament House, Canberra ACT 2600",0.935,jaro_winkler,auto,fuzzy,"Parliament House, Canberra ACT 2600",True
2,sydney opera house,"Sydney Opera House, Bennelong Point, Sydney NS...",0.872,jaro_winkler,review,fuzzy,"Sydney Opera House, Bennelong Point, Sydney NS...",True
3,101 collins st melbourne,"101 Collins Street, Melbourne VIC 3000",0.896,jaro_winkler,review,fuzzy,"101 Collins Street, Melbourne VIC 3000",True
4,Flinders st station melbourne,"Flinders Street Station, Melbourne VIC 3000",0.876,jaro_winkler,review,fuzzy,"Flinders Street Station, Melbourne VIC 3000",True
5,Federation sqr swanston street,"Federation Square, Swanston Street, Melbourne ...",0.869,jaro_winkler,review,fuzzy,"Federation Square, Swanston Street, Melbourne ...",True


### 4. Country names

Mostly typos plus one abbreviation (`USA`) that no edit-distance scorer will catch — abbreviations are an explicit-mapping problem, not fuzzy. Worth seeing the failure mode rather than papering over it.

In [8]:
run_scenario("country_names")

,input,matched_to,score,best_method,decision,via,expected,correct
0,untied kingdum,United Kingdom,0.941,jaro_winkler,auto,fuzzy,United Kingdom,True
1,USA,Spain,0.689,jaro_winkler,unmatched,fuzzy,United States,False
2,south afrika,South Africa,0.967,jaro_winkler,auto,fuzzy,South Africa,True
3,New Zeland,New Zealand,0.962,jaro_winkler,auto,fuzzy,New Zealand,True
4,Italya,Italy,0.967,jaro_winkler,auto,fuzzy,Italy,True
5,Argintina,Argentina,0.919,jaro_winkler,auto,fuzzy,Argentina,True
6,Netherlnds,Netherlands,0.982,jaro_winkler,auto,fuzzy,Netherlands,True
7,sth korea,South Korea,0.879,jaro_winkler,review,fuzzy,South Korea,True


## Per-method comparison — full breakdown for each scenario

Pick a representative input from each scenario and score it against every reference entry with every method. Each table answers two questions at once:

- **Which candidate is the likely match?** (the green-heavy row)
- **Which method gives you the most signal for this kind of data?** (the column with the widest spread)

Rows are sorted by best-of-row, so the most-likely match floats to the top.

In [9]:
def compare_methods(value: str, scenario: str, top_n: int | None = None):
    """Score `value` against every candidate for `scenario` using every
    method in SCORERS. Return a styled DataFrame: rows = candidates
    (sorted by best-of-row), columns = methods, gradient = score strength."""
    reference = reference_df.query("scenario == @scenario")["value"].tolist()
    norm_value = normalise(value)

    rows = []
    for r in reference:
        norm_r = normalise(r)
        row = {"candidate": r}
        for name, scorer in SCORERS.items():
            row[name] = round(scorer(norm_value, norm_r), 3)
        rows.append(row)

    df = pd.DataFrame(rows).set_index("candidate")
    df = (df.assign(_best=df.max(axis=1))
            .sort_values("_best", ascending=False)
            .drop(columns="_best"))
    if top_n:
        df = df.head(top_n)
    return (df.style
              .background_gradient(cmap="Greens", axis=0)
              .format("{:.3f}")
              .set_caption(f"Input: {value!r}  ·  Scenario: {scenario.replace('_', ' ')}"))

### Customer names — `"Jhon Smith"`

In [10]:
compare_methods("Jhon Smith", "customer_names")

,levenshtein,jaro_winkler,jaccard_tokens,jaccard_ngrams
candidate,,,,
John Smith,0.800,0.970,0.333,0.571
Jon Smyth,0.800,0.907,0.000,0.500
Hiroshi Tanaka,0.214,0.638,0.000,0.000
Rajesh Kumar,0.167,0.572,0.000,0.043
Jose Hernandez,0.143,0.565,0.000,0.040
Anna Kowalski,0.077,0.562,0.000,0.000
Carlos Martinez,0.200,0.556,0.000,0.000
Fatima Al-Rashid,0.125,0.547,0.000,0.000
Min-jun Kim,0.091,0.518,0.000,0.158


### Company names — `"Commwealth Bank Aus"`

In [11]:
compare_methods("Commwealth Bank Aus", "company_names")

,levenshtein,jaro_winkler,jaccard_tokens,jaccard_ngrams
candidate,,,,
Commonwealth Bank of Australia,0.633,0.874,0.167,0.562
Westpac Banking Corporation,0.185,0.636,0.000,0.175
National Australia Bank Limited,0.290,0.547,0.167,0.225
Woolworths Group Limited,0.208,0.542,0.000,0.048
Coles Group Limited,0.105,0.531,0.000,0.081
Macquarie Group Limited,0.087,0.511,0.000,0.000
CSL Limited,0.158,0.509,0.000,0.032
Fortescue Metals Group Limited,0.167,0.497,0.000,0.042
Qantas Airways Limited,0.045,0.479,0.000,0.077


### Addresses — `"130 Swanson st melborne VIC"`

In [12]:
compare_methods("130 Swanson st melborne VIC", "addresses")

,levenshtein,jaro_winkler,jaccard_tokens,jaccard_ngrams
candidate,,,,
"130 Swanston Street, Melbourne VIC 3000",0.711,0.898,0.222,0.694
"101 Collins Street, Melbourne VIC 3000",0.486,0.682,0.100,0.422
"Federation Square, Swanston Street, Melbourne VIC 3000",0.462,0.658,0.091,0.469
"Flinders Street Station, Melbourne VIC 3000",0.405,0.655,0.100,0.396
"1 Martin Place, Sydney NSW 2000",0.200,0.614,0.000,0.188
"Southern Cross Station, 99 Spencer Street, Melbourne VIC 3008",0.373,0.588,0.077,0.317
"Parliament House, Canberra ACT 2600",0.176,0.573,0.000,0.089
"Sydney Opera House, Bennelong Point, Sydney NSW 2000",0.200,0.547,0.000,0.150
"Melbourne Cricket Ground, Brunton Avenue, Richmond VIC 3002",0.228,0.547,0.083,0.267


### Country names — `"untied kingdum"`

In [13]:
compare_methods("untied kingdum", "country_names")

,levenshtein,jaro_winkler,jaccard_tokens,jaccard_ngrams
candidate,,,,
United Kingdom,0.786,0.941,0.000,0.500
United States,0.357,0.632,0.000,0.167
India,0.214,0.612,0.000,0.050
New Zealand,0.214,0.537,0.000,0.038
South Korea,0.143,0.537,0.000,0.038
Ireland,0.214,0.536,0.000,0.045
Australia,0.143,0.516,0.000,0.000
Argentina,0.214,0.504,0.000,0.136
France,0.071,0.492,0.000,0.000


## What's next

- **Source:** [`examples/fuzzy_matching.py`](fuzzy_matching.py) — the single-purpose functions used here.
- **SQL version:** [`examples/fuzzy_matching.sql`](fuzzy_matching.sql) — the same five-stage pipeline in PostgreSQL.
- **Interactive tool:** [`web/index.html`](../web/index.html) — visualise how each algorithm reasons about a pair of strings.
- **Concepts:** [`web/concepts.html`](../web/concepts.html) — where fuzzy matching sits in the broader data-cleansing workflow.

Made by **Jennifer Payne**.  
[GitHub](https://github.com/majorpayne-2021) · [LinkedIn](https://www.linkedin.com/in/jenniferapayne25/)